# EDA — Estilo de preguntas por profesor (Rendir.ai)

**Objetivo:** dar evidencia cuantitativa al *wedge* del proyecto: **cada profesor tiene un estilo de preguntas de desarrollo repetitivo y reconocible**. Si los estilos difieren de forma sistemática entre docentes, vale la pena *modelar al profesor* y no "un curso genérico".

Esto alimenta las hipótesis de `docs/research/guion_entrevistas.md`:
- **H1** — los alumnos estudian sin saber el estilo del profe.
- **Moat** — dataset propio por *(profesor × curso)*.

> **Datos:** `data/preguntas_muestra.csv` es una **muestra anonimizada** (profes `Prof_A`/`Prof_B`). Reemplázala por exámenes reales recolectados (sin datos personales) a medida que crezca el dataset.

Entorno: conda env `geo` (Python 3.10). Solo usa `pandas` + `matplotlib` (ya instalados).

In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Carga robusta: funciona tanto desde la raíz del repo como desde notebooks/
candidatos = [Path('data/preguntas_muestra.csv'), Path('../data/preguntas_muestra.csv')]
csv_path = next((p for p in candidatos if p.exists()), None)
assert csv_path is not None, 'No se encontró data/preguntas_muestra.csv'

df = pd.read_csv(csv_path)
print(f'Cargadas {len(df)} preguntas desde {csv_path}')
df.head()

## 1. Vista general
¿Cuántas preguntas tenemos por profesor, curso y fuente?

In [ ]:
print('Por profesor:')
print(df['profesor'].value_counts(), '\n')
print('Por fuente:')
print(df['fuente'].value_counts(), '\n')
print('Por curso:')
print(df['curso'].value_counts())

## 2. Longitud de las preguntas
La extensión típica es parte del "estilo": un profe puede pedir preguntas largas y elaboradas, otro preguntas cortas y precisas.

In [ ]:
df['n_palabras'] = df['pregunta'].str.split().str.len()

print('Estadísticos de longitud (palabras) por profesor:')
print(df.groupby('profesor')['n_palabras'].describe()[['mean', 'min', 'max']])

df.boxplot(column='n_palabras', by='profesor', grid=False)
plt.title('Longitud de las preguntas por profesor')
plt.suptitle('')
plt.xlabel('Profesor')
plt.ylabel('N° de palabras')
plt.show()

## 3. Verbo de comando (taxonomía de Bloom)
El **verbo inicial** revela el nivel cognitivo que exige el profe: *explique/analice* (comprensión/análisis) vs *demuestre/derive* (aplicación formal). Es la señal más fuerte del estilo.

In [ ]:
df['verbo'] = df['pregunta'].str.split().str[0].str.lower().str.strip('.,;:')

tabla_verbos = pd.crosstab(df['verbo'], df['profesor'])
print('Verbo inicial × profesor:')
print(tabla_verbos)

tabla_verbos.plot(kind='barh')
plt.title('Verbo de comando por profesor')
plt.xlabel('N° de preguntas')
plt.ylabel('Verbo inicial')
plt.tight_layout()
plt.show()

## 4. Términos más frecuentes
Vocabulario recurrente (quitando stopwords). Da pistas de los temas y el registro de cada profe.

In [ ]:
STOPWORDS = {
    'de', 'la', 'el', 'en', 'y', 'a', 'un', 'una', 'el', 'los', 'las', 'del',
    'que', 'por', 'con', 'su', 'se', 'al', 'lo', 'como', 'cómo', 'para', 'un',
    'sobre', 'entre', 'cual', 'cuál', 'sus', 'o', 'e', 'es', 'un', 'mismo',
}

def top_terminos(textos, n=10):
    tokens = []
    for t in textos:
        tokens += [w for w in re.findall(r'[a-záéíóúñü]+', t.lower())
                   if w not in STOPWORDS and len(w) > 3]
    return Counter(tokens).most_common(n)

for prof, grupo in df.groupby('profesor'):
    print(f'\n== {prof} ==')
    for palabra, freq in top_terminos(grupo['pregunta']):
        print(f'  {palabra:<15} {freq}')

## 5. Conclusiones (borrador)

Completar con los datos reales. Con la muestra de ejemplo se observa que:

- **Prof_A** se inclina por verbos de comprensión/análisis (*explique*, *analice*) → preguntas conceptuales.
- **Prof_B** se inclina por verbos formales (*demuestre*, *compare*, *derive*) → preguntas matemáticas/formales.

**Implicancia para Rendir.ai:** los estilos son **separables y repetitivos** entre docentes → confirma el wedge de *modelar al profesor*. Esto se traduce en el prompt de `ai/prompts/generate.md`, que condiciona la generación al estilo del docente.

**Siguiente paso del dataset:** sumar más profesores/cursos (semilla del moat) y medir si un clasificador simple puede predecir el profesor a partir del estilo de la pregunta.